# Economy of Bits — Notebook 01: Inputs and feasibility

Companion paper to *Are Firms More Unequal Than People?* (JRFM, under review).

The first paper found that the entire rise in S&P 500 concentration sat in firms GICS
calls digital. Sector is a label, not a mechanism. This project replaces it with the two
properties *Machine, Platform, Crowd* says actually matter — **near-zero marginal cost**
and **demand-side (network) returns to scale** — and asks which one carries the result.

**This notebook estimates nothing.** It answers one question: does the data exist at the
coverage the design needs? Three gates in section 7 of `PLAN.md` could stop the project,
and two of them are decided in cell 9. Run it top to bottom and send me the zip.

**Before you start** — put `FRED_API_KEY`, `BEA_API_KEY` and `CENSUS_API_KEY` in Colab
Secrets (the key icon in the left sidebar), not in a cell, and upload `bits_inputs.zip`
to the file pane.

Runtime: about 30–45 minutes, most of it EDGAR in cells 6 and 7.

## 1 · Setup

In [ ]:
# =============================================================================
# CELL 1 - Environment, keys, output folder
# =============================================================================
import os, re, sys, json, time, glob, zipfile, pathlib, datetime, shutil, warnings
import numpy as np, pandas as pd, requests

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

RUN_TAG = "bits_01_inputs"
BASE = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path.cwd()
ROOT = BASE / f"economy_of_bits_{RUN_TAG}"
OUTD, CACHE = ROOT / "outputs", ROOT / "cache"
for d in (OUTD, CACHE):
    d.mkdir(parents=True, exist_ok=True)
saved_tables = []

def savetable(df, name):
    if df is None or not len(df):
        print(f"  skipped {name} (empty)"); return
    df.to_parquet(OUTD / f"{name}.parquet")
    df.to_csv(OUTD / f"{name}.csv", index=False)
    saved_tables.append(name)
    print(f"  saved {name}  ({len(df):,} rows)")

def cached(name, builder, refresh=False):
    p = CACHE / f"{name}.parquet"
    if p.exists() and not refresh:
        d = pd.read_parquet(p); print(f"  cache hit: {name} ({len(d):,} rows)"); return d
    d = builder()
    if d is not None and len(d):
        d.to_parquet(p); print(f"  cached {name} ({len(d):,} rows)")
    return d

# ---- keys -------------------------------------------------------------------
# Colab: put these under the key icon in the left sidebar (Secrets), NOT in a
# cell. A key pasted into a notebook travels with every copy of the notebook.
def get_key(name):
    v = os.getenv(name)
    if not v:
        try:
            from google.colab import userdata
            v = userdata.get(name)
        except Exception:
            v = None
    return v

FRED_KEY   = get_key("FRED_API_KEY")
BEA_KEY    = get_key("BEA_API_KEY")
CENSUS_KEY = get_key("CENSUS_API_KEY")

# EDGAR requires a descriptive User-Agent with a contact address and refuses
# anonymous traffic. This is a published requirement, not a workaround.
SEC_UA = "Boston University Metropolitan College research (spattna1@bu.edu)"
SEC_HEADERS = {"User-Agent": SEC_UA, "Accept-Encoding": "gzip, deflate"}

print(f"writing to: {ROOT}")
print("keys present:",
      {"FRED": bool(FRED_KEY), "BEA": bool(BEA_KEY), "CENSUS": bool(CENSUS_KEY)})
print("\nNotebook 01 estimates nothing. It answers one question -- does the data")
print("exist at the coverage the design needs -- before any effort goes into the")
print("design. Every costly error in the first paper came from acting on data")
print("that had not been audited.")

## 2 · Source reconnaissance

None of these hosts was reachable from the machine this notebook was written on, so every
endpoint below is **unverified until you run it**. If something fails here, stop and send
me the output — a source that fails is a source the design cannot use.

In [ ]:
# =============================================================================
# CELL 2 - SOURCE RECONNAISSANCE
#
# None of these hosts is reachable from the machine this notebook was written
# on, so every endpoint below is unverified until this cell runs. Read the
# output before running anything else: a source that fails here is a source the
# design cannot use, and it is better to learn that now than in cell 9.
# =============================================================================
def probe(label, url, params=None, headers=None, expect=None, timeout=45):
    rec = {"source": label, "url": url.split("?")[0], "ok": False,
           "status": None, "note": ""}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=timeout)
        rec["status"] = r.status_code
        body = r.text[:400]
        if r.status_code != 200:
            rec["note"] = body.replace("\n", " ")[:180]
        elif expect and expect not in r.text:
            # Search the WHOLE body. An earlier version checked only the first
            # 4,000 characters, which reported companyfacts as failing because
            # Apple's "dei" block pushes the "us-gaap" marker past the window.
            rec["note"] = f"200 but '{expect}' not in the response: {body[:140]}"
        else:
            rec["ok"] = True
            rec["note"] = f"{len(r.content):,} bytes"
    except Exception as e:
        rec["note"] = f"{type(e).__name__}: {str(e)[:150]}"
    print(f"  {'OK  ' if rec['ok'] else 'FAIL'}  {label:<34} {rec['status']}  {rec['note'][:110]}")
    return rec

print("=== probing every source the design depends on ===\n")
rows = []

# --- SEC EDGAR: the primary fundamentals source ------------------------------
rows.append(probe("SEC ticker->CIK map",
                  "https://www.sec.gov/files/company_tickers.json",
                  headers=SEC_HEADERS, expect="cik_str"))
rows.append(probe("SEC XBRL frames (Revenues CY2022)",
                  "https://data.sec.gov/api/xbrl/frames/us-gaap/Revenues/USD/CY2022.json",
                  headers=SEC_HEADERS, expect="\"data\""))
rows.append(probe("SEC companyfacts (Apple, CIK 320193)",
                  "https://data.sec.gov/api/xbrl/companyfacts/CIK0000320193.json",
                  headers=SEC_HEADERS, expect="\"facts\""))

# --- FRED --------------------------------------------------------------------
if FRED_KEY:
    rows.append(probe("FRED observations",
                      "https://api.stlouisfed.org/fred/series/observations",
                      params={"series_id": "GDP", "api_key": FRED_KEY,
                              "file_type": "json", "limit": 3},
                      expect="observations"))
else:
    print("  SKIP  FRED  (no key)")

# --- BEA ---------------------------------------------------------------------
# Two things are being tested separately: whether the key works at all, and
# whether the dataset the design wants is actually exposed through the API. The
# Digital Economy Satellite Account is published as spreadsheets and may not be.
if BEA_KEY:
    B = "https://apps.bea.gov/api/data"
    rows.append(probe("BEA dataset list", B,
                      params={"UserID": BEA_KEY, "method": "GETDATASETLIST",
                              "ResultFormat": "JSON"}, expect="Dataset"))
    rows.append(probe("BEA FixedAssets params", B,
                      params={"UserID": BEA_KEY, "method": "GETPARAMETERLIST",
                              "datasetname": "FixedAssets", "ResultFormat": "JSON"},
                      expect="Parameter"))
    rows.append(probe("BEA GDPbyIndustry params", B,
                      params={"UserID": BEA_KEY, "method": "GETPARAMETERLIST",
                              "datasetname": "GDPbyIndustry", "ResultFormat": "JSON"},
                      expect="Parameter"))
else:
    print("  SKIP  BEA  (no key)")

# --- Census ------------------------------------------------------------------
# BTOS carries AI-use rates by sector from 2023. The endpoint shape has changed
# once already, so the discovery document is probed alongside the data call.
if CENSUS_KEY:
    rows.append(probe("Census API discovery",
                      "https://api.census.gov/data.json", expect="dataset"))
    # The BTOS path is NOT guessed here. api.census.gov/data.json is the
    # authoritative catalogue and cell 2b searches it for the real endpoint.
    print("  ---   Census BTOS endpoint resolved from the catalogue in cell 2b")
else:
    print("  SKIP  Census  (no key)")

# --- Yahoo, via yfinance ------------------------------------------------------
try:
    import yfinance as yf
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance"],
                   check=True)
    import yfinance as yf
try:
    fi = yf.Ticker("MSFT").fast_info
    cap = getattr(fi, "market_cap", None)
    ok = bool(cap)
    rows.append({"source": "yfinance fast_info", "url": "yahoo", "ok": ok,
                 "status": 200 if ok else None,
                 "note": f"MSFT ${cap/1e12:.2f}tn" if ok else "no market cap"})
    print(f"  {'OK  ' if ok else 'FAIL'}  {'yfinance fast_info':<34} "
          f"{rows[-1]['note']}")
except Exception as e:
    rows.append({"source": "yfinance fast_info", "url": "yahoo", "ok": False,
                 "status": None, "note": f"{type(e).__name__}: {str(e)[:120]}"})
    print(f"  FAIL  yfinance fast_info  {type(e).__name__}")

recon = pd.DataFrame(rows)
savetable(recon, "source_reconnaissance")

print("\n--- what this means for the design ---")
sec_ok = recon[recon.source.str.startswith("SEC")].ok.all()
if not sec_ok:
    print("  STOP. EDGAR is the only source of ten years of fundamentals. If the")
    print("  ticker map or the frames endpoint failed, check the User-Agent first:")
    print("  EDGAR rejects anonymous traffic with 403 and a plain-text explanation.")
else:
    print("  EDGAR reachable. The fundamentals panel can be built (cells 5-8).")
for s in ("BEA", "Census"):
    sub = recon[recon.source.str.startswith(s)]
    if len(sub) and not sub.ok.all():
        print(f"  {s} partially or wholly unavailable. That costs H5 (the industry")
        print(f"  validation arm) but nothing in H1-H4, which are firm-level.")

### 2b · Re-probe the two apparent failures

Neither was a data problem. The companyfacts check looked in too small a window, and the BTOS path was a guess — this cell resolves it from the Census catalogue instead of guessing again.


In [ ]:
# =============================================================================
# CELL 2b - Re-probe the two "failures". Paste this after cell 2 and run it.
#
# Neither is a data problem.
#
# (1) SEC companyfacts returned 200 with a valid body -- look at what it printed:
#     {"cik":320193,"entityName":"Apple Inc.","facts":{"dei":{...
#     That IS a good response. My probe looked for the string "us-gaap" in the
#     first 4,000 characters, and Apple's "dei" block alone is longer than that,
#     so the marker sat past the window. The test was wrong, not the endpoint.
#
# (2) Census BTOS 404 means I guessed the path. The discovery document came back
#     at 5.2 MB, so rather than guess a second time, this cell searches it for
#     the real endpoint and prints what it finds.
# =============================================================================
# CONCEPTS is defined in cell 5, which has not run yet if you are pasting this
# straight after cell 2. Define it here if absent so the cell stands alone.
if "CONCEPTS" not in dir():
    CONCEPTS = {
        "revenue": ["RevenueFromContractWithCustomerExcludingAssessedTax",
                    "RevenueFromContractWithCustomerIncludingAssessedTax",
                    "Revenues", "SalesRevenueNet", "SalesRevenueGoodsNet"],
        "cogs": ["CostOfGoodsAndServicesSold", "CostOfRevenue",
                 "CostOfGoodsSold", "CostOfServices"],
        "gross_profit": ["GrossProfit"],
        "rnd": ["ResearchAndDevelopmentExpense"],
        "sgna": ["SellingGeneralAndAdministrativeExpense"],
        "operating_income": ["OperatingIncomeLoss"],
        "ppe_net": ["PropertyPlantAndEquipmentNet"],
        "assets": ["Assets"],
        "goodwill": ["Goodwill"],
        "intangibles": ["IntangibleAssetsNetExcludingGoodwill",
                        "FiniteLivedIntangibleAssetsNet"],
    }

print("=== (1) SEC companyfacts, checked properly ===")
js = sec_get("https://data.sec.gov/api/xbrl/companyfacts/CIK0000320193.json") \
      if "sec_get" in dir() else None
if js is None:
    r = requests.get("https://data.sec.gov/api/xbrl/companyfacts/CIK0000320193.json",
                     headers=SEC_HEADERS, timeout=90)
    js = r.json() if r.status_code == 200 else None

if js:
    taxonomies = list(js.get("facts", {}).keys())
    gaap = js.get("facts", {}).get("us-gaap", {})
    print(f"  entity      {js.get('entityName')}")
    print(f"  taxonomies  {taxonomies}")
    print(f"  us-gaap tags {len(gaap):,}")
    # The real question is not whether the endpoint works. It is whether the
    # tags this design needs are present, and with how many annual filings.
    print("\n  tags this design needs, as Apple actually files them:")
    for field, tags in CONCEPTS.items():
        found = []
        for t in tags:
            node = gaap.get(t)
            if node:
                n = sum(1 for o in node.get("units", {}).get("USD", [])
                        if o.get("form") == "10-K")
                found.append(f"{t} ({n} 10-K facts)")
        print(f"    {field:<17} {found[0] if found else 'NOT FILED BY THIS FIRM'}")
    print("\n  companyfacts is healthy. Cell 7's fallback will work.")
    SEC_OK = True
else:
    SEC_OK = False
    print("  companyfacts genuinely unreachable. Stop and tell me.")

print("\n=== (2) Census: find the real BTOS endpoint ===")
btos = pd.DataFrame()
try:
    r = requests.get("https://api.census.gov/data.json", timeout=120)
    r.raise_for_status()
    cat = r.json().get("dataset", [])
    print(f"  discovery document lists {len(cat):,} datasets")
    rows = []
    for d in cat:
        blob = " ".join(str(d.get(k, "")) for k in
                        ("title", "description", "keyword", "identifier"))
        if re.search(r"business trends and outlook|\bbtos\b", blob, re.I):
            dist = d.get("distribution", [{}])
            rows.append({
                "title": d.get("title", "")[:90],
                "vintage": d.get("c_vintage"),
                "url": (dist[0].get("accessURL") if dist else None),
                "variables": (d.get("c_variablesLink") or "")[:110],
            })
    btos = pd.DataFrame(rows)
    if len(btos):
        print(f"  {len(btos)} BTOS datasets found:\n")
        display(btos.sort_values("vintage", na_position="first"))
        cand = btos.dropna(subset=["url"]).url.tolist()
        print("\n  testing each endpoint for the AI-adoption variables:")
        for u in cand[:6]:
            try:
                rv = requests.get(u.rstrip("/") + "/variables.json", timeout=60)
                if rv.status_code != 200:
                    print(f"    {rv.status_code}  {u}"); continue
                v = rv.json().get("variables", {})
                ai = [k for k in v if re.search(r"ai|artificial", k, re.I)]
                print(f"    200  {u}")
                print(f"         {len(v)} variables; AI-related: {ai[:12]}")
            except Exception as e:
                print(f"    ERR  {u}  {type(e).__name__}")
    else:
        print("  No BTOS entry in the catalogue. It may be published only as")
        print("  downloadable tables. H5 then rests on BEA, which is fully")
        print("  reachable -- the AI-adoption overlay becomes descriptive")
        print("  context in the discussion rather than a tested hypothesis.")
except Exception as e:
    print(f"  {type(e).__name__}: {str(e)[:180]}")

if len(btos) and "savetable" in dir():
    savetable(btos, "census_btos_endpoints")

print("\n" + "-" * 70)
print("VERDICT")
print("-" * 70)
print(f"  SEC EDGAR   {'healthy -- all three endpoints work' if SEC_OK else 'BROKEN'}")
print("  FRED        healthy")
print("  BEA         healthy, all three calls returned")
print(f"  Census      {'endpoint located above' if len(btos) else 'BTOS not in the API'}")
print("\n  H1-H4 are firm-level and depend only on EDGAR plus the uploaded panel.")
print("  Nothing above threatens them. Continue to cell 3 and run through to 13.")

## 3 · The prior panel

The validated market caps from the JRFM paper, uploaded rather than rebuilt. The companion
paper has to stand on exactly the same numbers as the paper under review.

In [ ]:
# =============================================================================
# CELL 3 - The prior panel
#
# Upload bits_inputs.zip to the Colab file pane (folder icon, left sidebar)
# before running this. It holds the validated market-cap panel from the JRFM
# paper -- 10,111 firm-years, already corrected for survivorship, the split
# basis, dual-class listings and the Paramount price corruption.
#
# It is NOT rebuilt here. Rebuilding means re-downloading 550 tickers of daily
# prices and re-deriving corrections that are already validated, and any drift
# between the two papers' panels would be untraceable. The companion paper must
# stand on exactly the same market caps as the one under review.
# =============================================================================
def find_inputs():
    for z in sorted(glob.glob(str(BASE / "**" / "bits_inputs.zip"), recursive=True)) + \
             sorted(glob.glob("/content/bits_inputs.zip")):
        dest = CACHE / "inputs"
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        print(f"  extracted {pathlib.Path(z).name}")
        return dest
    for d in (BASE / "inputs", BASE, CACHE / "inputs"):
        if (pathlib.Path(d) / "sp500_marketcap_panel_clean.csv.gz").exists():
            return pathlib.Path(d)
    return None

SRC = find_inputs()
if SRC is None:
    raise FileNotFoundError(
        "bits_inputs.zip not found.\n"
        "Upload it to the Colab file pane (folder icon in the left sidebar), then\n"
        "re-run this cell. It contains sp500_marketcap_panel_clean.csv.gz plus the\n"
        "three result tables from the JRFM paper that later cells compare against.")

panel = pd.read_csv(SRC / "sp500_marketcap_panel_clean.csv.gz")
panel["year"] = panel.year.astype(int)

PAPER_YEARS = list(range(2016, 2026))       # the JRFM estimation window
est = panel[panel.year.isin(PAPER_YEARS)].copy()

print(f"\nfull panel      {len(panel):,} firm-years, {panel.ticker.nunique()} tickers, "
      f"{panel.year.min()}-{panel.year.max()}")
print(f"paper window    {len(est):,} firm-years, {est.ticker.nunique()} tickers, "
      f"{min(PAPER_YEARS)}-{max(PAPER_YEARS)}")

# Reproduce two headline numbers as a check that the right file was uploaded.
chk = est.groupby("year").mktcap.agg(["size", "sum"])
chk["top1_share"] = est.groupby("year").mktcap.apply(
    lambda s: s.nlargest(max(1, int(round(0.01 * len(s))))).sum() / s.sum())
print("\nintegrity check against the JRFM results:")
print(f"  2016 top-1% share  {100*chk.top1_share.loc[2016]:.1f}%   (paper: 11.3%)")
print(f"  2025 top-1% share  {100*chk.top1_share.loc[2025]:.1f}%   (paper: 30.0%)")
ok = abs(100*chk.top1_share.loc[2016] - 11.3) < 0.2 and \
     abs(100*chk.top1_share.loc[2025] - 30.0) < 0.2
print("  ->", "matches" if ok else
      "DOES NOT MATCH. Wrong file, or the panel has been altered. Stop here.")
assert ok, "uploaded panel does not reproduce the published figures"

TICKERS = sorted(est.ticker.dropna().unique())
print(f"\n{len(TICKERS)} distinct tickers need fundamentals.")

# Sector labels, carried through so cell 12 can test whether bit intensity is
# just the GICS split under another name.
sectors = (est[["ticker", "sector"]].dropna().drop_duplicates("ticker")
           if "sector" in est.columns else pd.DataFrame(columns=["ticker", "sector"]))
DIGITAL_SECTORS = {"Information Technology", "Communication Services"}
DIGITAL_EXTRA = {"AMZN", "TSLA", "NFLX", "BKNG", "EBAY", "ABNB", "UBER", "DASH", "SHOP"}
sectors["gics_digital"] = (sectors.sector.isin(DIGITAL_SECTORS)
                           | sectors.ticker.str.upper().isin(DIGITAL_EXTRA))
print(f"GICS digital: {int(sectors.gics_digital.sum())} of {len(sectors)} firms")

## 4 · Ticker → CIK

In [ ]:
# =============================================================================
# CELL 4 - Ticker to CIK
#
# EDGAR is keyed on the Central Index Key, not the ticker. The mapping is
# many-to-one: GOOGL and GOOG are both CIK 1652044, which is the same dual-class
# problem the first paper solved on the market-cap side, arriving here from a
# different direction. Fundamentals are filed once per CIK, so a firm with two
# share classes must not have its revenue counted twice.
# =============================================================================
def fetch_cik_map():
    r = requests.get("https://www.sec.gov/files/company_tickers.json",
                     headers=SEC_HEADERS, timeout=60)
    r.raise_for_status()
    d = pd.DataFrame(r.json()).T
    d.columns = [c.lower() for c in d.columns]
    d = d.rename(columns={"cik_str": "cik", "title": "sec_name"})
    d["ticker"] = d.ticker.str.upper().str.strip()
    d["cik"] = d.cik.astype(int)
    return d[["ticker", "cik", "sec_name"]]

cik_map = cached("sec_cik_map", fetch_cik_map)
print(f"  SEC publishes {len(cik_map):,} ticker->CIK pairs")

# Yahoo writes class shares with a dash (BRK-B); EDGAR uses no separator or a
# dot. Try the ticker as-is, then the two rewrites, before giving up.
def variants(t):
    t = t.upper()
    out = [t]
    if "-" in t:
        out += [t.replace("-", ""), t.replace("-", ".")]
    return list(dict.fromkeys(out))

lookup = dict(zip(cik_map.ticker, cik_map.cik))
rows, unmatched = [], []
for t in TICKERS:
    hit = next((lookup[v] for v in variants(t) if v in lookup), None)
    if hit is None:
        unmatched.append(t)
    else:
        rows.append({"ticker": t, "cik": int(hit)})
tick2cik = pd.DataFrame(rows)

print(f"\nmatched   {len(tick2cik)} of {len(TICKERS)} tickers")
print(f"unmatched {len(unmatched)}")
if unmatched:
    # Expected: firms that have since been acquired or delisted are dropped from
    # the SEC's current ticker file even though they filed during the window.
    # They are not a bug, but they ARE the survivorship channel reappearing, so
    # cell 9 reports coverage by year rather than in aggregate.
    print(f"  {unmatched[:25]}")
    print("  These filed during the window but are absent from the CURRENT SEC")
    print("  ticker file, which lists live registrants only. Their fundamentals")
    print("  are recoverable by CIK but not by ticker; cell 9 quantifies the loss.")

CIKS = sorted(tick2cik.cik.unique())
print(f"\n{len(CIKS)} distinct CIKs (vs {len(tick2cik)} tickers -- the difference is")
print("dual-class firms filing once for two listings).")
savetable(tick2cik, "ticker_to_cik")

## 5 · What to pull from EDGAR, and why

In [ ]:
# =============================================================================
# CELL 5 - The XBRL concepts, and why each one is here
#
# US GAAP does not have one revenue tag. It has about six, and which a filer
# uses depends on its industry, its auditor and the year -- ASC 606 moved most
# large filers onto RevenueFromContractWithCustomer* in 2018-19, mid-sample.
# Taking a single tag would silently drop a third of the panel and, worse, would
# drop it non-randomly.
#
# So every field is a PRIORITY LIST. The first tag that yields a value for a
# firm-year wins, and the tag actually used is recorded, so cell 9 can report
# how much of the panel rests on each fallback.
# =============================================================================
CONCEPTS = {
    "revenue": [
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "RevenueFromContractWithCustomerIncludingAssessedTax",
        "Revenues",
        "SalesRevenueNet",
        "SalesRevenueGoodsNet",
        "RevenueFromContractWithCustomerExcludingAssessedTaxMember",
    ],
    "cogs": [
        "CostOfGoodsAndServicesSold",
        "CostOfRevenue",
        "CostOfGoodsSold",
        "CostOfServices",
    ],
    "gross_profit":  ["GrossProfit"],
    "rnd":           ["ResearchAndDevelopmentExpense"],
    "sgna":          ["SellingGeneralAndAdministrativeExpense"],
    "operating_income": ["OperatingIncomeLoss"],
    "ppe_net":       ["PropertyPlantAndEquipmentNet"],
    "assets":        ["Assets"],
    "goodwill":      ["Goodwill"],
    "intangibles":   ["IntangibleAssetsNetExcludingGoodwill",
                      "FiniteLivedIntangibleAssetsNet"],
}

# Flow concepts are reported for a period (CY2020); stock concepts for an
# instant (CY2020Q4I). Asking for the wrong frame returns 404, which is the
# single easiest way to conclude a field "has no data" when it has plenty.
STOCKS = {"ppe_net", "assets", "goodwill", "intangibles"}

YEARS = list(range(2015, 2026))   # one year before the window, for lags

print("fields the bit-intensity index needs:\n")
for f, tags in CONCEPTS.items():
    kind = "stock" if f in STOCKS else "flow"
    print(f"  {f:<17} {kind:<6} {len(tags)} tag(s)  {tags[0]}")
print(f"\nyears: {min(YEARS)}-{max(YEARS)}  ({len(YEARS)} years, one lag year included)")
print(f"calls if frames work: ~{sum(len(v) for v in CONCEPTS.values()) * len(YEARS)}")
print(f"calls if we fell back to companyfacts: {len(CIKS)}")

## 6 · EDGAR, the fast path

One call per concept-year returns every filer at once — roughly 80 calls instead of 550.

In [ ]:
# =============================================================================
# CELL 6 - EDGAR client
#
# EDGAR's published limit is 10 requests/second. The throttle below is
# deliberately slower. Exceeding it earns a temporary IP block, and on Colab
# that IP is shared with other notebooks, so the block may arrive without this
# notebook having misbehaved -- which is why every failure is recorded rather
# than raised.
# =============================================================================
SEC_SLEEP = 0.15
_sec_log = []

def sec_get(url, tries=3, timeout=60):
    """Returns parsed JSON, or None. Never raises: a missing concept-year is a
    404 by design, and a run of 550 firms will produce hundreds of them."""
    for attempt in range(tries):
        try:
            time.sleep(SEC_SLEEP)
            r = requests.get(url, headers=SEC_HEADERS, timeout=timeout)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 404:
                _sec_log.append({"url": url, "status": 404}); return None
            if r.status_code in (403, 429):
                wait = 5 * (attempt + 1)
                print(f"    {r.status_code} from EDGAR, backing off {wait}s")
                time.sleep(wait); continue
            _sec_log.append({"url": url, "status": r.status_code}); return None
        except Exception as e:
            if attempt == tries - 1:
                _sec_log.append({"url": url, "status": type(e).__name__})
                return None
            time.sleep(2 * (attempt + 1))
    return None

def frame_url(tag, year, is_stock):
    period = f"CY{year}Q4I" if is_stock else f"CY{year}"
    return f"https://data.sec.gov/api/xbrl/frames/us-gaap/{tag}/USD/{period}.json"

def fetch_frames(concepts, years, ciks):
    """One call per (tag, year) returns every filer reporting that concept --
    about 80 calls for the whole panel instead of 550 companyfacts downloads.

    Frames carry only filers whose fiscal period aligns with the calendar frame,
    so firms on a non-December year-end fall out here and are picked up by the
    companyfacts pass in cell 7. That is a coverage gap with a KNOWN shape, which
    is the kind worth having.
    """
    want = set(int(c) for c in ciks)
    rows, tried = [], 0
    for field, tags in concepts.items():
        is_stock = field in STOCKS
        for year in years:
            for tag in tags:
                tried += 1
                js = sec_get(frame_url(tag, year, is_stock))
                if not js or "data" not in js:
                    continue
                hits = 0
                for rec in js["data"]:
                    cik = int(rec.get("cik", -1))
                    if cik in want and rec.get("val") is not None:
                        rows.append({"cik": cik, "year": year, "field": field,
                                     "tag": tag, "value": float(rec["val"]),
                                     "source": "frames"})
                        hits += 1
                if hits:
                    print(f"    {field:<17} {year}  {tag[:44]:<44} {hits:>4} firms")
        print(f"  {field}: done")
    print(f"\n  {tried} frame calls, {len(rows):,} observations")
    return pd.DataFrame(rows)

frames_raw = cached("edgar_frames_raw",
                    lambda: fetch_frames(CONCEPTS, YEARS, CIKS))

if frames_raw is None or not len(frames_raw):
    print("\n!! No frame data at all. Do not continue to cell 7 -- check the")
    print("   reconnaissance output in cell 2 first. If the frames endpoint")
    print("   returned 200 there and nothing here, the CIK list is the problem.")
else:
    print(f"\n{len(frames_raw):,} observations covering "
          f"{frames_raw.cik.nunique()} of {len(CIKS)} CIKs")
    cov = (frames_raw.groupby(["field", "year"]).cik.nunique()
           .unstack("year").fillna(0).astype(int))
    print("\nfirms per field per year, from frames alone:")
    display(cov)

## 7 · EDGAR, the complete path

Firms on a non-December fiscal year are invisible to calendar frames. Apple closes in
September, Microsoft in June, Walmart in January. This pass picks them up.

In [ ]:
# =============================================================================
# CELL 7 - companyfacts fallback for the firms frames missed
#
# Frames only carry filers whose fiscal year matches the calendar frame. Apple
# closes in late September, Microsoft in June, Walmart in January -- none of
# them appears in a CY frame for a flow concept. That is a large and
# systematically non-random hole: it selects on fiscal calendar, which
# correlates with industry.
#
# companyfacts returns every fact a filer has ever tagged, with start and end
# dates, so a fiscal year can be assigned properly. Slower (one call per firm)
# but complete. Run it for every CIK the frames pass did not fully cover.
# =============================================================================
def assign_fy(end_date, months):
    """Assign a fiscal period to a calendar year: the year containing most of
    it. A period ending 2021-01-31 belongs to fiscal 2020, not 2021."""
    e = pd.Timestamp(end_date)
    mid = e - pd.Timedelta(days=int(30.4 * months / 2))
    return int(mid.year)

def parse_companyfacts(js, cik):
    out = []
    facts = (js or {}).get("facts", {}).get("us-gaap", {})
    for field, tags in CONCEPTS.items():
        is_stock = field in STOCKS
        for rank, tag in enumerate(tags):
            node = facts.get(tag)
            if not node:
                continue
            for unit, obs in node.get("units", {}).items():
                if unit != "USD":
                    continue
                for o in obs:
                    # 10-K only: quarterly frames would double-count, and the
                    # annual figure is what the ratios are defined on.
                    if o.get("form") not in ("10-K", "10-K/A", "20-F", "40-F"):
                        continue
                    if o.get("val") is None or not o.get("end"):
                        continue
                    if is_stock:
                        year = pd.Timestamp(o["end"]).year
                        months = 0
                    else:
                        if not o.get("start"):
                            continue
                        days = (pd.Timestamp(o["end"]) - pd.Timestamp(o["start"])).days
                        if not (330 <= days <= 400):     # annual periods only
                            continue
                        months = 12
                        year = assign_fy(o["end"], months)
                    out.append({"cik": int(cik), "year": year, "field": field,
                                "tag": tag, "tag_rank": rank,
                                "value": float(o["val"]),
                                "end": o["end"], "fy": o.get("fy"),
                                "accn": o.get("accn"), "source": "companyfacts"})
    return out

def fetch_companyfacts(ciks):
    rows, missing = [], []
    for i, cik in enumerate(ciks, 1):
        js = sec_get(f"https://data.sec.gov/api/xbrl/companyfacts/"
                     f"CIK{int(cik):010d}.json")
        if js is None:
            missing.append(cik)
        else:
            rows.extend(parse_companyfacts(js, cik))
        if i % 50 == 0:
            print(f"    {i}/{len(ciks)} CIKs, {len(rows):,} facts, "
                  f"{len(missing)} with no data")
    if missing:
        print(f"  no companyfacts for {len(missing)} CIKs: {missing[:12]}")
    return pd.DataFrame(rows)

# Which CIKs still need work? A firm is "covered" only if the two fields the
# index cannot be built without -- revenue and assets -- are present for it.
if frames_raw is not None and len(frames_raw):
    have = (frames_raw[frames_raw.field.isin(["revenue", "assets"])]
            .groupby("cik").field.nunique())
    covered = set(have[have >= 2].index)
else:
    covered = set()
NEED = [c for c in CIKS if c not in covered]
print(f"frames covered {len(covered)} CIKs; {len(NEED)} need companyfacts")
print(f"estimated time: {len(NEED) * (SEC_SLEEP + 0.35) / 60:.0f} minutes\n")

facts_raw = cached("edgar_companyfacts_raw", lambda: fetch_companyfacts(NEED))
print(f"\n{len(facts_raw):,} facts from companyfacts across "
      f"{facts_raw.cik.nunique() if len(facts_raw) else 0} CIKs")

## 8 · One value per firm-year

In [ ]:
# =============================================================================
# CELL 8 - Resolve to one value per firm-year-field
#
# A firm can report the same concept several times for one year: an original
# 10-K, a restatement, and the prior-year comparative inside the NEXT year's
# filing. All three are legitimate facts with the same (cik, year, field).
# Picking arbitrarily would mix restated and as-filed figures across the panel.
#
# Rule, applied in order:
#   1. prefer the higher-priority tag (the first that appears in CONCEPTS)
#   2. among those, prefer the LATEST accession -- the most recent statement of
#      that year, which is the restated figure where one exists
#   3. companyfacts beats frames, because it carries the dates that made rule 2
#      possible
# =============================================================================
parts = []
if frames_raw is not None and len(frames_raw):
    f = frames_raw.copy()
    f["tag_rank"] = [CONCEPTS[fl].index(tg) for fl, tg in zip(f.field, f.tag)]
    f["end"] = pd.NaT; f["accn"] = ""
    parts.append(f)
if facts_raw is not None and len(facts_raw):
    parts.append(facts_raw)
allfacts = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
if not len(allfacts):
    raise RuntimeError("No EDGAR facts at all. Stop and re-read cells 2, 6, 7.")

allfacts["src_rank"] = (allfacts.source == "frames").astype(int)   # 0 = companyfacts
allfacts["end"] = pd.to_datetime(allfacts.get("end"), errors="coerce")

resolved = (allfacts
            .sort_values(["cik", "year", "field", "tag_rank", "src_rank",
                          "end", "accn"],
                         ascending=[True, True, True, True, True, False, False])
            .groupby(["cik", "year", "field"], as_index=False)
            .first()[["cik", "year", "field", "value", "tag", "source"]])

print(f"{len(allfacts):,} raw facts -> {len(resolved):,} resolved firm-year-fields")
dupes = len(allfacts) - len(resolved)
print(f"{dupes:,} duplicate statements collapsed "
      f"({100*dupes/max(len(allfacts),1):.0f}% of the raw facts)")

wide = resolved.pivot_table(index=["cik", "year"], columns="field",
                            values="value", aggfunc="first").reset_index()
wide.columns.name = None
for c in CONCEPTS:
    if c not in wide.columns:
        wide[c] = np.nan

# Gross profit is reported by some filers and derivable for the rest. Derive it
# only where BOTH inputs exist, and record which firm-years were derived --
# a derived margin and a reported one are not quite the same object.
wide["gross_profit_reported"] = wide.gross_profit.notna()
wide["gross_profit"] = wide.gross_profit.fillna(wide.revenue - wide.cogs)
print(f"\ngross profit: {int(wide.gross_profit_reported.sum()):,} reported, "
      f"{int(wide.gross_profit.notna().sum() - wide.gross_profit_reported.sum()):,} "
      f"derived as revenue - COGS, "
      f"{int(wide.gross_profit.isna().sum()):,} unavailable")

fund = wide.merge(tick2cik, on="cik", how="left")
fund = fund[fund.year.between(min(YEARS), max(YEARS))]
print(f"\nfundamentals panel: {len(fund):,} rows, {fund.cik.nunique()} CIKs, "
      f"{fund.year.min()}-{fund.year.max()}")
savetable(resolved, "edgar_facts_resolved")

# Which tag actually carried each field? If the panel rests on a fallback for a
# large share of firm-years, that is a fact about the panel, not a detail.
tagmix = (resolved.groupby(["field", "tag"]).size().rename("n")
          .reset_index().sort_values(["field", "n"], ascending=[True, False]))
tagmix["share"] = tagmix.n / tagmix.groupby("field").n.transform("sum")
print("\ntag actually used, by field:")
display(tagmix[tagmix.share > 0.02].round(3))
savetable(tagmix, "edgar_tag_mix")

## 9 · The coverage audit

**The cell this notebook exists for.** Gates 1 and 2 are decided here.

In [ ]:
# =============================================================================
# CELL 9 - THE COVERAGE AUDIT.  This is the cell the notebook exists for.
#
# Section 7 of the plan names three conditions that would make the paper not
# worth writing. Two of them are decided here, before any modelling effort is
# spent. Read this output carefully; it is meant to be capable of stopping the
# project.
# =============================================================================
mc = est[["year", "ticker", "mktcap"]].copy()
merged = mc.merge(fund, on=["ticker", "year"], how="left")
print(f"market-cap firm-years in the window: {len(mc):,}")
print(f"of which matched to EDGAR fundamentals: "
      f"{int(merged.revenue.notna().sum()):,} "
      f"({100*merged.revenue.notna().mean():.1f}%)\n")

FIELDS = ["revenue", "cogs", "gross_profit", "rnd", "ppe_net", "assets"]
cov = merged.groupby("year")[FIELDS].apply(lambda d: d.notna().mean())
cov["n_firms"] = merged.groupby("year").ticker.nunique()
print("coverage by year (share of that year's firms with the field):")
display((100 * cov[FIELDS]).round(1).assign(n_firms=cov.n_firms))
savetable(cov.reset_index(), "fundamentals_coverage_by_year")

# --- gate 1: is the index buildable? -----------------------------------------
# The bit-intensity index needs gross margin, R&D intensity and asset lightness.
# R&D is the binding one: firms that do no R&D correctly report NOTHING rather
# than zero, so a missing R&D value is ambiguous between "did not spend" and
# "did not report". Treating all blanks as zero would push every non-R&D firm to
# the same point and manufacture the separation H1 is supposed to test.
core = ["revenue", "cogs", "assets", "ppe_net"]
core_ok = merged[core].notna().all(axis=1).mean()
rnd_ok = merged.rnd.notna().mean()
print(f"\ngate 1  core fields (revenue, COGS, assets, PP&E) complete: "
      f"{100*core_ok:.1f}%   {'PASS' if core_ok >= 0.70 else 'FAIL -- redesign'}")
print(f"        R&D reported: {100*rnd_ok:.1f}% of firm-years")
print("        R&D blanks are ambiguous between 'spent nothing' and 'did not")
print("        report'. Notebook 02 must handle them explicitly, not fill zeros.")

# --- gate 2: is bit intensity just the GICS split renamed? -------------------
d = merged.merge(sectors[["ticker", "gics_digital"]], on="ticker", how="left")
d["gross_margin"] = d.gross_profit / d.revenue
d["asset_light"] = 1 - (d.ppe_net / d.assets)
d["rnd_int"] = d.rnd / d.revenue
prev = d[d[["gross_margin", "asset_light"]].notna().all(axis=1)].copy()
if len(prev) > 100 and prev.gics_digital.notna().any():
    prev["provisional_bi"] = prev.groupby("year")[["gross_margin", "asset_light"]] \
        .transform(lambda s: (s - s.mean()) / s.std()).mean(axis=1)
    r = prev[["provisional_bi", "gics_digital"]].astype(float).corr().iloc[0, 1]
    print(f"\ngate 2  corr(provisional bit intensity, GICS digital) = {r:+.3f}")
    print(f"        {'PASS' if abs(r) < 0.90 else 'FAIL'} -- above 0.90 the paper has")
    print("        renamed a variable rather than explained one.")
    print("\n        provisional bit intensity by GICS group:")
    display(prev.groupby("gics_digital").provisional_bi.describe()[
        ["count", "mean", "std", "25%", "50%", "75%"]].round(3))
    print("        Note this is only TWO of the four components -- R&D intensity")
    print("        and revenue per employee are not in it. It is a gate, not the")
    print("        index, and notebook 02 builds the real one.")

# --- what fell out, and whether it fell out at random ------------------------
missing = merged[merged.revenue.isna()]
if len(missing):
    print(f"\n{len(missing):,} firm-years have market cap but no fundamentals.")
    print("by year:")
    print(missing.groupby("year").size().to_string())
    print("\nlargest firms among them (if these are big, the gap is not benign):")
    top = missing.nlargest(12, "mktcap")[["year", "ticker", "mktcap"]]
    top["cap_bn"] = (top.mktcap / 1e9).round(1)
    print(top[["year", "ticker", "cap_bn"]].to_string(index=False))
    lost = missing.mktcap.sum() / mc.mktcap.sum()
    print(f"\nthey are {100*lost:.2f}% of total market cap across the window.")
    print("If that share is concentrated in early years, it is the survivorship")
    print("channel again -- delisted firms stop filing -- and the paper must")
    print("report tail estimates on the matched subsample with that caveat.")
savetable(merged, "marketcap_fundamentals_merged")

## 10 · Employee counts

In [ ]:
# =============================================================================
# CELL 10 - Employees, from Yahoo
#
# Revenue per employee is the fourth component of the index and the one with the
# weakest source. EDGAR has no reliable employee-count tag: dei:EntityNumberOf
# Employees is optional and sparsely used. Yahoo carries a CURRENT headcount and
# no history.
#
# So this is a static firm attribute applied across years, which is wrong in a
# known direction: firms that grew headcount over the decade have their early
# revenue-per-employee understated. Notebook 02 will report the index with and
# without this component, and if the two disagree the component is dropped. It
# is collected here so that decision rests on evidence.
# =============================================================================
def fetch_employees(tickers):
    rows = []
    for i, t in enumerate(tickers, 1):
        n = None
        try:
            info = yf.Ticker(t).get_info()
            n = info.get("fullTimeEmployees")
        except Exception:
            pass
        rows.append({"ticker": t, "employees": float(n) if n else np.nan})
        if i % 100 == 0:
            got = sum(1 for r in rows if pd.notna(r["employees"]))
            print(f"    {i}/{len(tickers)}, {got} with a headcount")
        time.sleep(0.12)
    return pd.DataFrame(rows)

employees = cached("yahoo_employees", lambda: fetch_employees(TICKERS))
got = int(employees.employees.notna().sum())
print(f"\nheadcount for {got} of {len(TICKERS)} tickers "
      f"({100*got/len(TICKERS):.0f}%)")
if got:
    e = employees.dropna()
    print(f"median {e.employees.median():,.0f}, "
          f"range {e.employees.min():,.0f} to {e.employees.max():,.0f}")
savetable(employees, "employees_current_snapshot")

## 11 · Two-sidedness, coded before any result is seen

In [ ]:
# =============================================================================
# CELL 11 - Two-sidedness, coded in advance
#
# H2 turns on a hand-coded binary, which is the softest part of the design. The
# defence is procedural: the coding is written down BEFORE any tail index is
# estimated, every firm carries a one-line reason, and ambiguous cases are coded
# PIPELINE -- the conservative direction, which works against H2 rather than for
# it. Coding is fixed here and not revisited after seeing results.
#
# The list below was extended once, after checking which candidate tickers are
# actually in the panel and BEFORE any tail index existed. That is completing
# the coding, not tuning it -- but it is recorded here so a reader can judge.
#
# All three criteria must hold:
#   (1) the firm intermediates between two identifiable groups it prices
#       separately;
#   (2) participants on one side benefit as the other side grows;
#   (3) the firm sets the rules of interaction rather than owning the inventory.
#
# Criterion (3) is what excludes Netflix (licenses and owns content), Apple
# (revenue is overwhelmingly hardware it manufactures and sells) and Microsoft
# (software licences), all of which run platforms as a minority of the business.
# Coding them as platforms would make the label mean "large technology firm",
# which is the very conflation this paper exists to break.
# =============================================================================
PLATFORM = {
    # payment networks: cardholders on one side, merchants and acquirers on the other
    "V":    "card network; interchange priced separately to issuers and acquirers",
    "MA":   "card network; same two-sided structure as Visa",
    "AXP":  "card network and issuer; merchant discount and cardholder fees are separate",
    "PYPL": "payments platform intermediating buyers and merchants",
    "FISV": "Fiserv; merchant acquiring and issuer processing across both sides",
    "CPAY": "Corpay; closed-loop payment network between fleets and merchants",
    "GPN":  "merchant acquirer intermediating merchants and networks",
    # exchanges: the textbook two-sided market
    "CME":  "derivatives exchange; buyers and sellers, listing and trading fees separate",
    "ICE":  "exchange group; listings priced to issuers, access priced to traders",
    "NDAQ": "exchange; issuer listing fees and member trading fees are distinct sides",
    "CBOE": "options exchange; same structure",
    "MKTX": "MarketAxess; electronic bond market between dealers and institutions",
    "COIN": "crypto exchange; matches buyers and sellers, holds no inventory",
    # advertising marketplaces: users on one side, advertisers on the other
    "GOOGL": "search and YouTube; users unpriced, advertisers priced, classic cross-subsidy",
    "GOOG":  "second share class of the same firm",
    "META":  "social platform; users unpriced, advertisers priced",
    "TTD":   "demand-side ad platform intermediating buyers and inventory owners",
    # marketplaces for goods and services
    "AMZN": "third-party marketplace and AWS; sets rules for sellers it does not own",
    "EBAY": "pure marketplace; no inventory",
    "ETSY": "pure marketplace; no inventory",
    "UBER": "riders and drivers priced separately, cross-side effects in both directions",
    "ABNB": "guests and hosts; the firm owns no property",
    "DASH": "diners, merchants and couriers; three sides priced separately",
    "BKNG": "travel marketplace; inventory belongs to hotels",
    "EXPE": "travel marketplace; same structure",
    "LYV":  "Ticketmaster intermediates venues and fans with separate fee structures",
    "TRIP": "travel meta-search; travellers unpriced, suppliers and advertisers priced",
}
# Coded PIPELINE despite running a platform as part of the business. The reason
# is recorded because a reviewer will ask about each of these by name.
AMBIGUOUS_PIPELINE = {
    "AAPL": "App Store is two-sided but revenue is overwhelmingly owned hardware",
    "MSFT": "Azure marketplace and LinkedIn sit inside a software licensing business",
    "NFLX": "single-sided subscription; licenses and produces the inventory",
    "CRM":  "SaaS licence revenue; AppExchange is marginal",
    "MTCH": "network effects are within one side, not across two priced sides",
    "MCD":  "franchising resembles a platform but the firm owns brand and supply",
    "MAR":  "franchising and management; same reasoning as McDonald's",
    "SPGI": "ratings and indices sold to one side",
}

plat = pd.DataFrame({"ticker": list(PLATFORM), "platform": True,
                     "reason": list(PLATFORM.values())})
amb = pd.DataFrame({"ticker": list(AMBIGUOUS_PIPELINE), "platform": False,
                    "reason": ["AMBIGUOUS, coded pipeline: " + v
                               for v in AMBIGUOUS_PIPELINE.values()]})
coding = pd.concat([plat, amb], ignore_index=True)
coding = coding[coding.ticker.isin(TICKERS)]

n_plat = int(coding.platform.sum())
print(f"pre-registered coding covers {len(coding)} of {len(TICKERS)} firms: "
      f"{n_plat} platform, {len(coding)-n_plat} ambiguous-but-pipeline")

# --- gate 3 -------------------------------------------------------------------
print(f"\ngate 3  platform firms so far: {n_plat}   "
      f"(PROVISIONAL -- the coding is completed in notebook 02)")
print(f"        Threshold is 25. Currently "
      f"{'above' if n_plat >= 25 else 'below'} it, with "
      f"{len(TICKERS) - coding.ticker.nunique()} firms still uncoded, so the")
print("        gate is not decided yet. It is decided once the review queue")
print("        below has been worked through -- which happens before any tail")
print("        index is computed, not after.")
print("        Below about 25 the Hill estimator has a tail of 2-3 and H2 cannot")
print("        be tested at any useful precision. If the final count falls short,")
print("        H2 becomes a comparison of central tendency and the plan says so.")

# --- the review queue ---------------------------------------------------------
# Everything not yet coded, largest first. Uncoded firms default to pipeline,
# so a large uncoded platform biases against H2 -- but it is still an error, and
# this list is how it gets found. Coding is completed in notebook 02, still
# before any tail index is computed.
last = est[est.year == est.year.max()][["ticker", "mktcap"]]
todo = (last[~last.ticker.isin(coding.ticker)]
        .sort_values("mktcap", ascending=False).head(40).copy())
todo["cap_bn"] = (todo.mktcap / 1e9).round(1)
print(f"\n{len(last) - len(coding.ticker.unique())} firms are still uncoded. "
      f"The 40 largest, for review in notebook 02:")
print(todo[["ticker", "cap_bn"]].to_string(index=False))
savetable(coding, "platform_coding_preregistered")
savetable(todo[["ticker", "cap_bn"]], "platform_coding_review_queue")

## 12 · Industry and macro sources

In [ ]:
# =============================================================================
# CELL 12 - Industry and macro sources (H5)
#
# This arm validates the firm-level bit-intensity index against an independent
# industry measure built from national accounts. It is the only hypothesis that
# depends on BEA and Census, so a failure here costs H5 and nothing else.
#
# Everything is wrapped: the point of the cell is to find out what these APIs
# actually return, not to assume it. Whatever comes back is saved raw so the
# next iteration can parse it properly.
# =============================================================================
macro = {}

# --- BEA ----------------------------------------------------------------------
# Intellectual property products (software, R&D, entertainment originals) as a
# share of private fixed assets, by industry. This is Haskel and Westlake's
# intangible-capital measure taken straight from the national accounts, and it
# is the industry analogue of firm-level bit intensity.
if BEA_KEY:
    B = "https://apps.bea.gov/api/data"
    def bea(**kw):
        p = {"UserID": BEA_KEY, "ResultFormat": "JSON", **kw}
        try:
            r = requests.get(B, params=p, timeout=90)
            js = r.json().get("BEAAPI", {})
            if "Error" in js or "Error" in js.get("Results", {}):
                err = js.get("Error") or js["Results"]["Error"]
                print(f"    BEA error: {str(err)[:200]}")
                return None
            return js.get("Results", {})
        except Exception as e:
            print(f"    BEA {type(e).__name__}: {str(e)[:150]}")
            return None

    print("BEA: listing datasets")
    res = bea(method="GETDATASETLIST")
    if res and "Dataset" in res:
        names = [d["DatasetName"] for d in res["Dataset"]]
        print(f"  {len(names)} datasets: {names}")
        macro["bea_datasets"] = pd.DataFrame(res["Dataset"])
        # Is the Digital Economy account exposed at all? It is normally
        # published as spreadsheets, so a negative answer here is expected
        # and simply means H5 uses IPP capital instead.
        dig = [n for n in names if "digital" in n.lower()]
        print(f"  digital-economy dataset exposed: {dig if dig else 'no -- use IPP capital'}")

    print("\nBEA: what parameters does FixedAssets take?")
    res = bea(method="GETPARAMETERLIST", datasetname="FixedAssets")
    if res and "Parameter" in res:
        print(pd.DataFrame(res["Parameter"])[["ParameterName", "ParameterDescription"]]
              .to_string(index=False)[:900])
        print("\nBEA: available FixedAssets tables")
        res2 = bea(method="GETPARAMETERVALUES", datasetname="FixedAssets",
                   ParameterName="TableName")
        if res2 and "ParamValue" in res2:
            tv = pd.DataFrame(res2["ParamValue"])
            macro["bea_fixedassets_tables"] = tv
            hits = tv[tv.apply(lambda r: r.astype(str).str.contains(
                "intellectual|property|industry", case=False).any(), axis=1)]
            print(f"  {len(tv)} tables; {len(hits)} mention IPP or industry:")
            print(hits.head(15).to_string(index=False)[:1200])
            print("\n  Pick the right table from this list in notebook 05 rather")
            print("  than guessing a table code here.")
else:
    print("BEA: no key, H5 industry arm unavailable")

# --- Census -------------------------------------------------------------------
# BTOS reports AI-use rates by sector from September 2023. It is short, but it
# is the only firm-level AI-adoption measure that is public and current, and it
# is what connects this paper to the JRFM future-scope section.
if CENSUS_KEY:
    print("\nCensus: BTOS availability")
    try:
        r = requests.get("https://api.census.gov/data/timeseries/eits/btos"
                         "/variables.json", timeout=60)
        if r.status_code == 200:
            v = pd.DataFrame(r.json().get("variables", {})).T
            macro["census_btos_variables"] = v.reset_index()
            print(f"  {len(v)} BTOS variables. First 20:")
            print(list(v.index)[:20])
        else:
            print(f"  variables.json -> {r.status_code}. BTOS may have moved; the")
            print("  discovery document at https://api.census.gov/data.json lists")
            print("  current endpoints, and notebook 05 should read it.")
    except Exception as e:
        print(f"  {type(e).__name__}: {str(e)[:150]}")
else:
    print("\nCensus: no key")

# --- FRED ---------------------------------------------------------------------
FRED_SERIES = {
    "Y033RC1Q027SBEA": "private fixed investment: intellectual property products",
    "A008RC1Q027SBEA": "private fixed investment: equipment",
    "CP":              "corporate profits after tax",
    "GDP":             "nominal GDP",
    "NCBEILQ027S":     "nonfinancial corporate equity outstanding",
}
if FRED_KEY:
    print("\nFRED:")
    rows = []
    for sid, label in FRED_SERIES.items():
        try:
            r = requests.get("https://api.stlouisfed.org/fred/series/observations",
                             params={"series_id": sid, "api_key": FRED_KEY,
                                     "file_type": "json"}, timeout=60)
            if r.status_code != 200:
                print(f"  FAIL {sid:<16} {r.status_code}"); continue
            d = pd.DataFrame(r.json()["observations"])
            d["date"] = pd.to_datetime(d.date)
            d["value"] = pd.to_numeric(d.value.replace(".", np.nan), errors="coerce")
            d = d.dropna(subset=["value"])
            d["series_id"] = sid; d["label"] = label
            rows.append(d[["series_id", "label", "date", "value"]])
            print(f"  OK   {sid:<16} {len(d):>5} obs  "
                  f"{d.date.min().date()} to {d.date.max().date()}  {label}")
        except Exception as e:
            print(f"  FAIL {sid:<16} {type(e).__name__}")
    if rows:
        macro["fred"] = pd.concat(rows, ignore_index=True)
else:
    print("\nFRED: no key")

for name, df in macro.items():
    savetable(df, f"macro_{name}")
print(f"\n{len(macro)} macro/industry tables captured: {list(macro)}")

## 13 · Gates, manifest, download

In [ ]:
# =============================================================================
# CELL 13 - Manifest, gate summary, zip, download
#
# The three gates from section 7 of the plan are restated here in one place. If
# any of them fails, notebook 02 does not get written as designed -- the design
# changes first. That is the whole purpose of running an inputs notebook before
# a modelling notebook.
# =============================================================================
files = sorted(p for p in OUTD.rglob("*") if p.is_file())
rows = []
for p in files:
    row = {"file": str(p.relative_to(OUTD)), "kb": round(p.stat().st_size / 1024, 1)}
    if p.suffix == ".parquet":
        try:
            row["rows"] = len(pd.read_parquet(p))
        except Exception:
            row["rows"] = None
    rows.append(row)
manifest = pd.DataFrame(rows)
manifest.to_csv(OUTD / "MANIFEST.csv", index=False)
display(manifest)

print("\n" + "=" * 72)
print("GATE SUMMARY -- the three conditions from section 7 of the plan")
print("=" * 72)

g1 = float(merged[["revenue", "cogs", "assets", "ppe_net"]].notna().all(axis=1).mean())
g1_pass = g1 >= 0.70
print(f"\n1. EDGAR coverage of the core fields   {100*g1:5.1f}%   "
      f"{'PASS' if g1_pass else 'FAIL'}   (need >= 70%)")
if not g1_pass:
    print("   The bit-intensity index is too thin. Options for notebook 02:")
    print("   restrict to the matched subsample and report the selection, or")
    print("   drop the weakest component and rebuild the index on three.")

try:
    g2 = abs(float(r)); g2_pass = g2 < 0.90
    print(f"\n2. |corr(bit intensity, GICS digital)| {g2:5.3f}   "
          f"{'PASS' if g2_pass else 'FAIL'}   (need < 0.90)")
    if not g2_pass:
        print("   Bit intensity is the sector split under another name. The paper")
        print("   would be renaming a variable, not explaining one. Reconsider.")
except NameError:
    g2_pass = None
    print("\n2. correlation gate did not run (cell 9 had too little data)")

g3 = n_plat; g3_pass = None   # provisional: coding is finished in notebook 02
print(f"\n3. Platform firms coded so far         {g3:5d}     "
      f"PROVISIONAL   (threshold 25, coding completes in nb 02)")
if g3 < 25:
    print("   H2 becomes a comparison of central tendency, not of tails. That is")
    print("   still publishable, but the hypothesis must be restated honestly.")

gates = [g1_pass, g2_pass, g3_pass]
print("\n" + "-" * 72)
if all(g is not False for g in gates):
    print("All gates passed. Notebook 02 can proceed as planned:")
    print("  - finish the platform coding from the review queue in cell 11")
    print("  - build the four-component bit-intensity index, within-year z-scored")
    print("  - decide the revenue-per-employee component with and against")
    print("  - validate against the GICS split, then hand off to notebook 03")
else:
    print("At least one gate failed. Send me this output before writing more code.")
    print("The design changes first; that is what this notebook is for.")
print("-" * 72)

summary = {
    "run": RUN_TAG,
    "when": datetime.datetime.now().isoformat(timespec="seconds"),
    "market_cap_firm_years": int(len(mc)),
    "matched_to_edgar": int(merged.revenue.notna().sum()),
    "match_rate": round(float(merged.revenue.notna().mean()), 4),
    "ciks": int(len(CIKS)),
    "unmatched_tickers": unmatched,
    "gate1_core_coverage": round(g1, 4),
    "gate2_corr_with_gics": (round(float(r), 4) if g2_pass is not None else None),
    "gate3_platform_firms": int(g3),
    "gates_passed": [bool(x) if x is not None else None for x in gates],
    "tables": saved_tables,
}
(OUTD / "run_summary.json").write_text(json.dumps(summary, indent=2, default=str))

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
archive = shutil.make_archive(str(BASE / f"economy_of_bits_{RUN_TAG}_{stamp}"),
                              "zip", root_dir=OUTD.parent, base_dir=OUTD.name)
print(f"\nzipped -> {archive}  "
      f"({pathlib.Path(archive).stat().st_size/1024:.0f} KB)")
try:
    from google.colab import files as colab_files
    colab_files.download(archive)
    print("Download started. Allow pop-ups if the browser blocked it.")
except ImportError:
    print("Not on Colab. Copy the archive from the path above.")